## Volatility Filtering for Optimal Timing for Train and Inference

In [1]:
import pandas as pd
import numpy as np

## Data Loader
## Input to this NB is the generated csv file from data_handler.py given Ticker

In [2]:
df = pd.read_csv("RL_Data.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df = df.sort_values('timestamp').reset_index(drop=True) # date sort
#df['log_return'] = np.log(df['close'] / df['close'].shift(1))  # percentage->amplify for GARCH fitting
df = df.dropna(subset=['log_return']).reset_index(drop=True)
df = df[['timestamp','date','ask_price_1', 'ask_size_1','bid_price_1','bid_size_1',
         'ask_price_2', 'ask_size_2','bid_price_2','bid_size_2',
         'ask_price_3', 'ask_size_3','bid_price_3','bid_size_3',
         'ask_price_4', 'ask_size_4','bid_price_4','bid_size_4',
         'ask_price_5', 'ask_size_5','bid_price_5','bid_size_5',
         'open','high','low','close',
         'volume','volatility','log_return']]

In [3]:
## Optimal Timing for training
df_sorted = df.sort_values(['date', 'volatility'])
df_filtered = df_sorted.groupby('date').head(80).reset_index(drop=True)
daily_volume = df_filtered.groupby('date')['volume'].sum().reset_index()
daily_volume.rename(columns={'volume': 'daily_volume'}, inplace=True)
df_filtered = df_filtered.merge(daily_volume, on='date', how='left')
df_filtered.head()

,timestamp,date,ask_price_1,ask_size_1,bid_price_1,bid_size_1,ask_price_2,ask_size_2,bid_price_2,bid_size_2,...,bid_price_5,bid_size_5,open,high,low,close,volume,volatility,log_return,daily_volume
0,2023-09-12 15:24:00+00:00,2023-09-12,176.43,20100,176.51,500,176.43,107600.0,176.51,40500.0,...,176.51,27600.0,176.47,176.51,176.43,176.43,13475.0,0.000865,-0.000283,1211098.0
1,2023-09-12 15:25:00+00:00,2023-09-12,176.36,20000,176.45,24800,176.36,10000.0,176.45,61100.0,...,176.45,61100.0,176.44,176.45,176.36,176.37,14430.0,0.000943,-0.000340,1211098.0
2,2023-09-12 15:26:00+00:00,2023-09-12,176.36,42500,176.42,3000,176.36,47500.0,176.42,10900.0,...,176.42,3000.0,176.37,176.42,176.35,176.37,12824.0,0.000943,0.000000,1211098.0
3,2023-09-12 15:27:00+00:00,2023-09-12,176.38,37500,176.47,31600,176.38,22500.0,176.47,10000.0,...,176.47,45200.0,176.39,176.47,176.37,176.46,9822.0,0.000998,0.000510,1211098.0
4,2023-09-12 15:20:00+00:00,2023-09-12,176.30,20000,176.37,10000,176.30,24000.0,176.37,12500.0,...,176.37,30000.0,176.30,176.37,176.30,176.35,17686.0,0.001039,0.000227,1211098.0


In [4]:
df_filtered = df_filtered[['timestamp','ask_price_1', 'ask_size_1','bid_price_1','bid_size_1',
         'ask_price_2', 'ask_size_2','bid_price_2','bid_size_2',
         'ask_price_3', 'ask_size_3','bid_price_3','bid_size_3',
         'ask_price_4', 'ask_size_4','bid_price_4','bid_size_4',
         'ask_price_5', 'ask_size_5','bid_price_5','bid_size_5',
         'open','high','low','close','daily_volume',
         'volume','volatility','log_return']]
df_filtered.to_csv("RL_Data_opt_timing.csv",index=False)

## This approach first decision-making on optimal timing
## Then throw the sizing part to RL agent for training.
## As for inference, we need to forecast ask1 price/size, bid1 price/ size, open, volume, daily volume, and volatility

In [5]:
steps_forward = 390 # one trading day forward
df.set_index('timestamp', inplace=True)
df['time'] = df.index.time
df['date'] = df.index.date
last_date = df['date'].max()
next_date = last_date + pd.Timedelta(days=1)
while next_date.weekday() >= 5:  # 5 = Saturday, 6 = Sunday
    next_date += pd.Timedelta(days=1)

start_time = pd.Timestamp(f"{next_date} 13:30:00")
end_time = pd.Timestamp(f"{next_date} 19:59:00")

df_forecast = pd.DataFrame({
    'timestamp': pd.date_range(start=start_time, periods=390, freq='1min'),
    'ask_price_1': np.zeros(steps_forward,dtype=float),
    'ask_size_1' : np.zeros(steps_forward,dtype=int),
    'bid_price_1' : np.zeros(steps_forward,dtype=float),
    'bid_size_1' : np.zeros(steps_forward,dtype=int),
    'open': np.zeros(steps_forward,dtype=float),
    'volume': np.zeros(steps_forward,dtype=int),
    'volatility': np.zeros(steps_forward,dtype=float),
    'daily_volume' : np.zeros(steps_forward,dtype=float),
})
df_forecast.set_index('timestamp', inplace=True)
df_forecast['time'] = df_forecast.index.time
df_forecast['date'] = df_forecast.index.date

In [6]:
from datetime import timedelta
def non_recurrent_hist_mean(train,test,col,window_size):
    train_dates = sorted(train['date'].unique())
    forecast=[]
    conf_lower=[]
    conf_upper=[]
    for idx, row in test.iterrows():
        time_to_forecast = row['time']
        date_to_forecast = row['date']
        past_dates = [date_to_forecast - timedelta(days=x) for x in range(1, window_size + 1)]
        past_dates = [d for d in past_dates if d.weekday() < 5 and d in train_dates] # exclude weekend
        past_data = train[(train['time'] == time_to_forecast) & (train['date'].isin(past_dates))][col]
        historical_mean = past_data.mean()
        if np.isnan(historical_mean):
            historical_mean = train['bid'].iloc[-1] # last close price
        forecast.append(historical_mean)
    forecast=np.array(forecast)
    return forecast

ask_price_1_forecast = non_recurrent_hist_mean(df, df_forecast, 'ask_price_1', 3)
ask_size_1_forecast = non_recurrent_hist_mean(df, df_forecast, 'ask_size_1', 3).astype(int)
import matplotlib.pyplot as plt

In [7]:
df_forecast['ask_price_1'] = ask_price_1_forecast
df_forecast['ask_size_1'] = ask_size_1_forecast
df_forecast['bid_price_1'] = non_recurrent_hist_mean(df, df_forecast, 'bid_price_1', 3)
df_forecast['bid_size_1'] = non_recurrent_hist_mean(df, df_forecast, 'bid_size_1', 3).astype(int)
df_forecast['open'] = non_recurrent_hist_mean(df, df_forecast, 'open', 3)
df_forecast['volume'] = non_recurrent_hist_mean(df, df_forecast, 'volume', 3).astype(int)
df_forecast['volatility'] = non_recurrent_hist_mean(df, df_forecast, 'volatility', 3)
df_forecast = df_forecast.reset_index()

## We pick 20 low volatility as optimal timing

In [8]:
optimal_timing=df_forecast.loc[df_forecast['volatility']<=np.percentile(df_forecast['volatility'].values, 5),:] # we keep 5% for optimal timing
optimal_timing

,timestamp,ask_price_1,ask_size_1,bid_price_1,bid_size_1,open,volume,volatility,daily_volume,time,date
99,2024-09-23 15:09:00,230.37,3000,230.48,26000,230.410,19163,0.001340,0.0,15:09:00,2024-09-23
100,2024-09-23 15:10:00,230.30,1000,230.42,20000,230.410,27847,0.001114,0.0,15:10:00,2024-09-23
167,2024-09-23 16:17:00,231.26,10000,231.36,10000,231.280,15045,0.001236,0.0,16:17:00,2024-09-23
174,2024-09-23 16:24:00,231.38,10000,231.48,34000,231.470,15810,0.001356,0.0,16:24:00,2024-09-23
175,2024-09-23 16:25:00,231.41,10000,231.53,17000,231.460,17959,0.001340,0.0,16:25:00,2024-09-23
176,2024-09-23 16:26:00,231.44,29000,231.61,14500,231.530,21271,0.001346,0.0,16:26:00,2024-09-23
181,2024-09-23 16:31:00,231.65,20000,231.74,5000,231.720,17331,0.001276,0.0,16:31:00,2024-09-23
182,2024-09-23 16:32:00,231.56,10000,231.70,15000,231.655,20030,0.001356,0.0,16:32:00,2024-09-23
183,2024-09-23 16:33:00,231.57,72400,231.63,12900,231.580,11889,0.001179,0.0,16:33:00,2024-09-23
184,2024-09-23 16:34:00,231.59,6700,231.66,10000,231.600,6917,0.000981,0.0,16:34:00,2024-09-23


In [14]:
optimal_timing=optimal_timing.copy()
optimal_timing['daily_volume'] = optimal_timing['volume'].sum()
optimal_timing = optimal_timing[['timestamp','ask_price_1', 'ask_size_1','bid_price_1','bid_size_1',
                                'open','volume','daily_volume','volatility']]
optimal_timing

,timestamp,ask_price_1,ask_size_1,bid_price_1,bid_size_1,open,volume,daily_volume,volatility
99,2024-09-23 15:09:00,230.37,3000,230.48,26000,230.410,19163,457932,0.001340
100,2024-09-23 15:10:00,230.30,1000,230.42,20000,230.410,27847,457932,0.001114
167,2024-09-23 16:17:00,231.26,10000,231.36,10000,231.280,15045,457932,0.001236
174,2024-09-23 16:24:00,231.38,10000,231.48,34000,231.470,15810,457932,0.001356
175,2024-09-23 16:25:00,231.41,10000,231.53,17000,231.460,17959,457932,0.001340
176,2024-09-23 16:26:00,231.44,29000,231.61,14500,231.530,21271,457932,0.001346
181,2024-09-23 16:31:00,231.65,20000,231.74,5000,231.720,17331,457932,0.001276
182,2024-09-23 16:32:00,231.56,10000,231.70,15000,231.655,20030,457932,0.001356
183,2024-09-23 16:33:00,231.57,72400,231.63,12900,231.580,11889,457932,0.001179
184,2024-09-23 16:34:00,231.59,6700,231.66,10000,231.600,6917,457932,0.000981


In [15]:
optimal_timing.to_csv("RL_Data_opt_timing_inference.csv",index=False)

<h1> RL Model for picking optimal bet size</h1>
<h2>Training</h2>
<h3>The input to this block is a CSV with optimal timings.</h3>

In [5]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt


from GordonRitter.utils.bet_sizing_model_utils import BetSizingModel
from GordonRitter.utils.bet_sizing_env import TradingEnvironment

#Data collection

data = pd.read_csv("RL_Data_opt_timing_inference.csv") 

state_columns = ['bid_price_1', 'bid_size_1', 'open', 'volume','daily_volume', 'volatility']

#OPTIMAL TIMINGS PER DAY
optm_pts_per_day = 20

# Unifying the type of numerical columns 
float_cols = data.select_dtypes(include=['float','int']).columns
data[float_cols] = data[float_cols].astype(np.float32)

#Handling missing values
rows_with_nan = data.isna().any(axis=1)
data.loc[rows_with_nan] = data.ffill().loc[rows_with_nan]
for col in data.columns:
    if data[col].isna().any():
        print(col)

#80% of the data is set aside for the training.
train_split = (int(data.shape[0]*0.8)//optm_pts_per_day)*optm_pts_per_day

data_train = data.iloc[:train_split]
data_test = data.iloc[train_split:]


# Min-Max normalization of float features.
data_min = data_train[float_cols].min()
data_max = data_train[float_cols].max()

for col in float_cols:
    data_train[col] = (data_train[col] - data_min[col])/(data_max[col]-data_min[col])
    data_test[col] = (data_test[col] - data_min[col])/(data_max[col]-data_min[col])


#Enviroment prep
env = TradingEnvironment(
    data=data_train,
    reset_count=optm_pts_per_day,
    initial_inventory=10000, # INVENTORY SIZE
    state_columns=state_columns
)

model = BetSizingModel(name='test1', env=env)

#Training the model
model.train(10000) #SAMPLES TO BE USED FOR TRAINING.

/tmp/ipykernel_54396/393306549.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_train[col] = (data_train[col] - data_min[col])/(data_max[col]-data_min[col])
/tmp/ipykernel_54396/393306549.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_test[col] = (data_test[col] - data_min[col])/(data_max[col]-data_min[col])
/home/malhar/Workspace/Blockhouse/Blockhouse-ML/blockhouse_ml/equities/sell/utils/data_handler.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a

Training from scratch...


<h2>Model Inference</h2>

In [6]:
env = TradingEnvironment(
    data=data_test,
    reset_count=optm_pts_per_day,
    initial_inventory=10000,
    state_columns=state_columns
)

model.test(env=env, steps=int(data_test.shape[0]*0.95))
for action_list in env.tradelist:
# Each row tells how much of the total inventory
# to sell for the corresponding optimal of a day.

# THE RATIO SHOULD BE USED WITH CAPPING, LIMITING TOTAL
# TRADED VOLUME TO THE INVENTORY.  
    print(action_list)

/home/malhar/Workspace/Blockhouse/Blockhouse-ML/blockhouse_ml/equities/sell/utils/data_handler.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['mid_price'] = (self.data['high'] + self.data['low']) / 2
/home/malhar/Workspace/Blockhouse/Blockhouse-ML/blockhouse_ml/equities/sell/utils/data_handler.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data["mean_vol"] = self.data['mid_price'].pct_change().rolling(window=window_size).mean()
/home/malhar/Workspace/Blockhouse/Blockhouse-ML/block

[0.0, 0.5359386801719666, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0915442705154419, 0.0, 0.9156397581100464]
[0.0, 0.0, 0.0, 0.7577047944068909, 0.0, 0.0, 0.0, 0.0, 0.0, 0.009934604167938232, 0.0, 0.2774854302406311]
[0.756639301776886, 0.0, 0.2390347719192505, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.04432767629623413]
[0.0, 0.0, 0.0, 0.02229940891265869, 0.0, 0.3253552317619324, 0.0, 0.08456993103027344, 0.0, 0.9807868599891663]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.7769777774810791, 0.0, 0.0, 1.0]
[0.0, 0.00875246524810791, 0.0, 0.0, 0.0, 0.5577006340026855, 0.0, 0.40579187870025635, 0.0, 0.0, 0.0, 0.0, 0.20845454931259155]
[0.0, 0.0, 0.014348328113555908, 0.0, 0.0, 0.0, 0.7988471984863281, 0.0, 0.0, 0.0, 0.3839157223701477]
[0.05111473798751831, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.8005333542823792]
[0.0, 0.03880131244659424, 0.0, 0.18796586990356445, 0.0, 0.7995172142982483]
[0.